# Crypto Data Prep — point-in-time universe, dead tokens included

**Standalone.** This touches nothing in the equities pipeline. It writes exactly one new file,
`crypto_market_data_2018_2026.parquet`, and reads nothing from the S&P 500 project.

Companion to `crypto_train.ipynb`. Same scientific discipline as the equities work: the goal is a
**corrected, honest result**, not a winning Sharpe.

---

## What Stage 0 established (all empirically probed, not assumed)

**The survivorship problem is INVERTED relative to equities.** In the S&P 500 project, Yahoo refuses
to serve most fully-delisted tickers: 429 of 703 de-indexed names are unrecoverable, and coverage
drifts **+34.9pp** across the OOS window, so the level of that Sharpe is untrustworthy.

Binance does the opposite. It **retains delisted symbols and their full history**:

| Probe | Result |
|---|---|
| Dead pairs sampled | **15/15 served complete history** |
| LUNA | served all the way down to \$0.00005 |
| UST / FTT | served through the depeg / the FTX collapse |
| Archive vs live API | only **1** USDT pair (`NBTUSDT`) exists in the bulk archive but not `exchangeInfo` |
| USDT pairs | 701 → **242 dead (BREAK)** |
| BTC pairs | 488 → **439 dead** |

So the dead tokens are both **enumerable** and **retrievable**. The equities notebook's central
problem does not exist here.

**Crypto is genuinely 24/7** — BTCUSDT has a bar on **100.00%** of calendar days (3,257 bars over
3,257 days, zero gaps, 365–366/yr). This is why annualization is **365**, everywhere. It is measured,
not conventional.

---

## Two DEVIATIONS from the locked design — both confirmed before building

**1. Universe is ranked by trailing dollar volume, NOT market cap.**
No free point-in-time market-cap source exists. Probed and rejected: CoinGecko free (401, hard-capped
at 365 days), CoinPaprika (402, paid-only), CryptoCompare (401, key required), CoinCap v2 (DNS dead),
Messari (404), and CMC's `/historical/YYYYMMDD/` pages (HTTP 200 but a **client-side shell** — the
listing is not server-rendered, its internal `data-api` is bot-blocked, and scraping it would breach
their ToS). Market cap = price × circulating supply, and **supply history is the piece nobody gives
away free.**

So the universe is **top 100 by trailing 30d median dollar volume**, shifted to `t-1`. This is
fully point-in-time and leak-free, and it mirrors the equities project's own `SIZE` proxy (trailing
63d median dollar volume). **It is not market cap** — it over-ranks high-turnover meme coins and
under-ranks low-float assets. There is deliberately **no `market_cap` column** in the output: a
column by that name carrying a volume proxy is exactly the kind of thing that gets misread as truth
six months later. The ranking variable ships as `DollarVol30` and `UnivRank`.

**2. USD prices are cross-rated from BTC pairs before ~2020.**
Binance had **6** clean USDT coins on 2018-01-01 and did not reach 100 until ~Jul 2020. It was
BTC-quoted back then. Clean coins live at each year-start:

| Date | USDT-only | BTC-quoted | Union |
|---|---|---|---|
| 2018-01-01 | **6** | 96 | **97** |
| 2019-01-01 | 20 | 146 | 148 |
| 2020-01-01 | 84 | 168 | 175 |
| 2021-01-01 | 180 | 235 | 245 |
| 2022-01-01 | 299 | 308 | 336 |

A USDT-only 2018 start is impossible: 6 coins cannot form deciles. So a coin's USD price is taken
from its USDT pair when one trades that day, else from `coin/BTC × BTCUSDT`. **Declared
approximation:** the daily cross-rate is applied as a single per-day factor to O/H/L/C. The true
intraday USD high/low is unknowable from daily bars (the coin's BTC extreme and BTC's USD extreme
need not co-occur), so bar *extremes* are approximate while the *close* is exact. `Source` records
which path every row took, so this is auditable rather than buried.

---

## Two crypto-specific hazards that fabricate returns (§5)

**Redenomination** is the one with no equities escape hatch. Binance klines carry **no split
adjustment** — there is no `Adj Close`. `COCOS` did a 1:1000 token swap on 2021-01-23 (\$0.000695 →
\$0.9003 across a 4-day halt, volume ÷1000). Eight such events sit in this panel. They are detected
from the bars (a >5x jump *spanning a halt*) because no free adjustment factor exists.

## The ticker-reuse hazard (crypto-specific, and it will eat you)

`LUNAUSDT` collapses to \$0.00005 on 2022-05-13, halts **18 days**, then resumes at **\$8.87** on
2022-05-31 — that is **Terra 2.0, a different asset on the same ticker**. Naively concatenated it
produces a **+177,399x single-day return** that would single-handedly dominate a long-short book.
The old history is *not* under `LUNCUSDT` (which only begins 2022-09-09), so you cannot fix this by
switching symbols. `FTTUSDT` shows a **311-day gap** (delisted, then relisted).

**Fix:** a symbol is split into separate assets at any trading gap longer than `GAP_SPLIT_DAYS`
(§4). Extreme surviving moves are **reported, not silently filtered** — §8.

## Carried over from the equities pipeline, deliberately

- **No blanket `dropna()`.** Rows die only when OHLCV is unusable. Signal warm-up NaNs are KEPT: the
  CNN reads the image alone, and the scorer drops NaN signals per benchmark.
- Leveraged tokens (`*UP`/`*DOWN`/`BULL`/`BEAR`) are **derivatives, not tokens** — excluded.
  Stablecoins/fiat/gold have no price dynamics and would sit permanently in the LOW vol tercile —
  excluded. Wrapped duplicates (WBTC/BTCB/…) would double-count their underlying — excluded.

## 1. Imports & knobs

In [ ]:
import os, io, time, json, zipfile, urllib.request, urllib.error
import numpy as np
import pandas as pd
from concurrent.futures import ThreadPoolExecutor
from tqdm.auto import tqdm
import warnings; warnings.filterwarnings('ignore')

# ---------------- locked knobs (pre-registered; do not tune) ----------------
START_DATE   = '2018-01-01'      # sample start (locked)
END_DATE     = '2026-07-01'      # exclusive upper bound
TOP_N        = 100               # universe size (locked)
VOL_WIN      = 30                # realized-vol window, CALENDAR days (locked)
ANNUALIZER   = 365               # crypto is 24/7 -- verified 100.0% day coverage (locked)

# ---------------- derived-design knobs (stated, not tuned) ------------------
LIQ_WIN         = 30    # trailing window for the dollar-volume universe rank
GAP_SPLIT_DAYS  = 7     # trading gap that splits a symbol into a new asset (ticker reuse)
RATIO_SPLIT     = 5.0   # price jump ACROSS A HALT this large => units change (redenomination)
MIN_HISTORY     = 60    # skip symbols with under 60 daily bars ever

# Signal lookbacks. Crypto trades 365d/yr, so these are CALENDAR-day equivalents of the
# equities trading-day definitions (which were 252/21/5 trading days ~ 12mo/1mo/1wk).
# The locked vol window is likewise specified in *days*, which is what fixes this reading.
MOM_LB, MOM_SKIP = 365, 30   # 2-12 month momentum, skipping the last month
STR_LB           = 30        # 1-month reversal
WSTR_LB          = 7         # 1-week reversal
MA_LAGS          = [3, 5, 10, 20, 50, 100, 200]
MA_COLS          = [f'MA{L}' for L in MA_LAGS]

# ---------------- paths ----------------
BASE_URL  = 'https://data-api.binance.vision'   # public read-only mirror; no API key
CACHE_DIR = './crypto_cache'
OUT_FILE  = 'crypto_market_data_2018_2026.parquet'
os.makedirs(CACHE_DIR, exist_ok=True)

# ---------------- exclusions ----------------
STABLE_FIAT = {
    'USDC','BUSD','TUSD','DAI','FDUSD','USDP','SUSD','USTC','UST','FRAX','LUSD','XUSD','USD1',
    'PAXG','XAUT','EUR','EURI','AEUR','GBP','AUD','TRY','BRL','RUB','JPY','ZAR','ARS','PLN',
    'RON','CZK','MXN','COP','IDRT','NGN','UAH','VAI','USDS','USDSB','BIDR','DOGS',
}
WRAPPED_DUPES = {'WBTC','BTCB','WBETH','BETH','STETH','WSTETH','SOLBTC','RENBTC','HBTC','TBTC'}

def is_leveraged(base: str) -> bool:
    return base.endswith(('UP','DOWN')) or base.endswith(('BULL','BEAR'))

print(f"{START_DATE} -> {END_DATE} | top {TOP_N} by trailing {LIQ_WIN}d dollar volume")
print(f"annualization {ANNUALIZER} | vol window {VOL_WIN}d | gap-split {GAP_SPLIT_DAYS}d")
print(f"MOM {MOM_LB}/{MOM_SKIP}  STR {STR_LB}  WSTR {WSTR_LB} (calendar days)")
print(f"out -> {OUT_FILE}")

## 2. Universe enumeration — every symbol Binance EVER listed

`exchangeInfo` retains delisted symbols with `status='BREAK'`. That is the whole ballgame: it means
the dead tokens are enumerable from the live API, which is precisely what Yahoo refuses to do for
equities. We keep **both** `TRADING` and `BREAK`.

In [ ]:
def api_get(path, tries=4):
    for i in range(tries):
        try:
            req = urllib.request.Request(BASE_URL + path, headers={'User-Agent': 'research/1.0'})
            with urllib.request.urlopen(req, timeout=30) as r:
                return json.load(r)
        except Exception as e:
            if i == tries - 1:
                raise
            time.sleep(0.5 * (i + 1))

info = api_get('/api/v3/exchangeInfo')
allsym = pd.DataFrame([{'symbol': s['symbol'], 'base': s['baseAsset'],
                        'quote': s['quoteAsset'], 'status': s['status']}
                       for s in info['symbols']])
print(f"exchangeInfo: {len(allsym):,} symbols | TRADING {(allsym.status=='TRADING').sum():,} "
      f"| BREAK(dead) {(allsym.status=='BREAK').sum():,}")

cand = allsym[allsym['quote'].isin(['USDT', 'BTC'])].copy()
n0 = len(cand)
cand['excl'] = np.where(cand['base'].map(is_leveraged), 'leveraged',
               np.where(cand['base'].isin(STABLE_FIAT), 'stable/fiat',
               np.where(cand['base'].isin(WRAPPED_DUPES), 'wrapped-dupe', '')))
excl_counts = cand[cand.excl != ''].groupby(['excl']).size()
cand = cand[cand.excl == ''].drop(columns='excl')

print(f"\nUSDT/BTC-quoted symbols: {n0:,}")
for k, v in excl_counts.items():
    print(f"  excluded {k:14s}: {v:3d}")
print(f"  -> kept              : {len(cand):,} symbols across {cand['base'].nunique():,} base assets")
print(f"     USDT {(cand.quote=='USDT').sum():3d} ({(cand[cand.quote=='USDT'].status=='BREAK').sum()} dead) | "
      f"BTC {(cand.quote=='BTC').sum():3d} ({(cand[cand.quote=='BTC'].status=='BREAK').sum()} dead)")
SYMBOLS = sorted(set(cand['symbol']) | {'BTCUSDT'})   # BTCUSDT is both an asset and the cross-rate
print(f"\nto download: {len(SYMBOLS):,} symbol histories")

## 3. Download — daily klines, cached per symbol

Paginated 1000 bars at a time. Cached to `crypto_cache/` so a re-run costs nothing and the whole
notebook stays reproducible. Failures are counted and reported in §8, never hidden.

Kline fields: `0` openTime, `1` open, `2` high, `3` low, `4` close, `5` **base-asset volume**,
`7` quote-asset volume. We keep base volume as `Volume`, so `Close × Volume` is dollar volume —
the same construction the equities project uses for its size proxy.

In [ ]:
START_MS = int(pd.Timestamp(START_DATE).timestamp() * 1000)
END_MS   = int(pd.Timestamp(END_DATE).timestamp() * 1000)

def fetch_symbol(symbol):
    # Full daily history for one symbol, cached. Returns DataFrame or None.
    cache = os.path.join(CACHE_DIR, f'{symbol}.parquet')
    if os.path.exists(cache):
        return pd.read_parquet(cache)
    rows, t = [], 0                      # t=0 -> Binance returns from the listing date
    while True:
        try:
            d = api_get(f'/api/v3/klines?symbol={symbol}&interval=1d&startTime={t}&limit=1000')
        except Exception:
            return None
        if not d:
            break
        rows += d
        if len(d) < 1000:
            break
        t = d[-1][0] + 86_400_000
        time.sleep(0.05)
    if not rows:
        return None
    df = pd.DataFrame(rows).iloc[:, [0, 1, 2, 3, 4, 5]]
    df.columns = ['ts', 'Open', 'High', 'Low', 'Close', 'Volume']
    df['Date'] = pd.to_datetime(df['ts'], unit='ms', utc=True).dt.tz_localize(None).dt.normalize()
    for c in ['Open', 'High', 'Low', 'Close', 'Volume']:
        df[c] = df[c].astype(float)
    df = df.drop(columns='ts').drop_duplicates('Date').sort_values('Date')
    df.to_parquet(cache, index=False)
    return df

t0 = time.time()
raw = {}
failed = []
with ThreadPoolExecutor(max_workers=12) as ex:
    for sym, d in tqdm(zip(SYMBOLS, ex.map(fetch_symbol, SYMBOLS)), total=len(SYMBOLS),
                       desc='download'):
        if d is None or len(d) < MIN_HISTORY:
            failed.append(sym)
        else:
            raw[sym] = d

print(f"\ndownloaded {len(raw):,} symbols in {(time.time()-t0)/60:.1f} min "
      f"({sum(len(v) for v in raw.values()):,} bars)")
print(f"unusable (<{MIN_HISTORY} bars or fetch failed): {len(failed):,}")

## 4. USD panel — cross-rate the BTC-quoted pairs, then consolidate per asset

For each base asset and date we prefer the **USDT** print; where none exists (which is nearly
everything before mid-2020) we fall back to `coin/BTC × BTCUSDT_close`.

The cross-rate is one factor per day applied to the whole bar. The **close is exact**; the
**high/low are approximate**, because the coin's BTC extreme and BTC's USD extreme need not occur at
the same instant. Daily bars cannot resolve that, so it is declared here and recorded per row in
`Source` rather than hidden.

In [ ]:
btc_usd = raw['BTCUSDT'][['Date', 'Close']].rename(columns={'Close': 'BTCUSD'})
print(f"BTCUSDT cross-rate: {len(btc_usd):,} days | "
      f"{btc_usd.Date.min().date()} -> {btc_usd.Date.max().date()}")

meta = cand.set_index('symbol')[['base', 'quote']].to_dict('index')
frames = []
for sym, d in raw.items():
    m = meta.get(sym)
    if m is None:                       # BTCUSDT itself when not in cand (it is, but be safe)
        m = {'base': 'BTC', 'quote': 'USDT'} if sym == 'BTCUSDT' else None
        if m is None:
            continue
    d = d.copy()
    d['Symbol'] = m['base']
    if m['quote'] == 'USDT':
        d['Source'] = 'USDT'
        d['_prio'] = 0
    else:                               # BTC-quoted -> cross-rate into USD
        d = d.merge(btc_usd, on='Date', how='inner')
        for c in ['Open', 'High', 'Low', 'Close']:
            d[c] = d[c] * d['BTCUSD']
        d = d.drop(columns='BTCUSD')
        d['Source'] = 'BTCX'
        d['_prio'] = 1
    frames.append(d[['Date','Symbol','Open','High','Low','Close','Volume','Source','_prio']])

panel = pd.concat(frames, ignore_index=True)
before = len(panel)
panel = (panel.sort_values(['Symbol', 'Date', '_prio'])
              .drop_duplicates(['Symbol', 'Date'], keep='first')      # USDT wins over BTCX
              .drop(columns='_prio').reset_index(drop=True))
print(f"panel: {before:,} rows -> {len(panel):,} after USDT-preferred consolidation "
      f"| {panel.Symbol.nunique():,} assets")

# usable OHLCV only -- this is the ONLY row-dropping rule (no blanket dropna)
b = len(panel)
panel = panel[(panel[['Open','High','Low','Close']] > 0).all(axis=1)
              & panel[['Open','High','Low','Close','Volume']].notna().all(axis=1)]
panel = panel[(panel.Date >= START_DATE) & (panel.Date < END_DATE)].reset_index(drop=True)
print(f"dropped {b-len(panel):,} unusable/out-of-window rows -> {len(panel):,}")
print(f"\nsource mix: \n{panel.Source.value_counts().to_string()}")

## 5. Discontinuities — split assets at ticker reuse AND at redenominations

Two distinct traps, both of which fabricate enormous returns out of nothing.

**Ticker reuse.** `LUNAUSDT` = Terra Classic to 2022-05-13, then an 18-day halt, then **Terra 2.0 on
the same ticker** at \$8.87. Naively concatenated: **+177,399x** in a day.

**Redenomination (the crypto stock-split).** `COCOS` 2021-01-23: \$0.000695 → \$0.9003 across a
**4-day halt**, with volume falling ~1000x — a **1:1000 token swap**. Binance klines are **not
adjusted** for these, the way `Adj Close` adjusts equities for splits. There is no free adjustment
factor available, so this must be detected from the bars themselves. A 7-day gap rule alone misses
it: the halt was only 4 days.

**The discriminator is the halt, and it is clean.** Across the whole panel there are 19 jumps
beyond 5x. All **8** that coincide with a trading halt are units changes (COCOS 1:1000, DREP 1:100,
SUN, QUICK, BNX, PHB, SNM, BTCST — price ÷k *with volume ×k*). All **11** that occur on consecutive
days are real market events: LUNA's collapse, OM's April-2025 crash (volume ×200, dollar volume
×32), BCN, and genuine pumps.

So: **split on a long gap, or on a >5x jump that spans a halt.** Consecutive-day moves are never
touched — LUNA's −99.97% and OM's −84% survive intact, because those are exactly the real collapses
a short leg is supposed to capture. Deleting them would be laundering. This is a units correction,
**not** winsorization: no return is clipped, the series is cut where its units change.

In [ ]:
panel = panel.sort_values(['Symbol', 'Date']).reset_index(drop=True)
step  = panel.groupby('Symbol')['Date'].diff().dt.days
pr    = panel.groupby('Symbol')['Close'].pct_change() + 1.0      # price ratio vs previous bar

reuse   = (step > GAP_SPLIT_DAYS)                                 # long halt -> possibly a new asset
redenom = (step > 1) & ((pr > RATIO_SPLIT) | (pr < 1.0 / RATIO_SPLIT))   # halt + units-sized jump
brk = (reuse | redenom).fillna(False)
panel['_era'] = brk.groupby(panel['Symbol']).cumsum() + 1

print(f"discontinuities found: {int(brk.sum())}")
print(f"  long halts   (>{GAP_SPLIT_DAYS}d)          : {int(reuse.fillna(False).sum())}")
print(f"  redenominations (halt + >{RATIO_SPLIT:g}x) : {int((redenom & ~reuse).fillna(False).sum())}")
print("\nredenominations caught (price ratio across the halt):")
rd = panel[(redenom & ~reuse).fillna(False)].assign(price_ratio=pr[(redenom & ~reuse).fillna(False)],
                                                    halt_days=step[(redenom & ~reuse).fillna(False)])
if len(rd):
    print(rd[['Date','Symbol','Close','price_ratio','halt_days']]
          .to_string(index=False, float_format=lambda x: f"{x:,.4g}"))

n_eras = panel.groupby('Symbol')['_era'].nunique()
split = n_eras[n_eras > 1]
print(f"\nassets split into eras: {len(split)} / {panel.Symbol.nunique()}")

panel['Symbol'] = np.where(panel['Symbol'].isin(split.index),
                           panel['Symbol'] + '#' + panel['_era'].astype(str),
                           panel['Symbol'])
panel = panel.drop(columns='_era')
print(f"assets after era-splitting: {panel.Symbol.nunique():,}")

# ---- proof both traps are defused, and that REAL collapses are untouched ----
chk = panel.sort_values(['Symbol','Date'])
_st = chk.groupby('Symbol')['Date'].diff().dt.days
_r  = chk.groupby('Symbol')['Close'].pct_change()
chk = chk.assign(ret=_r, step=_st)
print("\nlargest surviving 1-bar returns anywhere in the panel:")
print(chk.nlargest(5, 'ret')[['Date','Symbol','Close','ret','step']]
      .to_string(index=False, float_format=lambda x: f"{x:,.4g}"))
print("\nreal collapses that MUST survive (consecutive-day, genuine):")
for s in ['LUNA#1', 'OM']:
    gg = chk[chk.Symbol == s]
    if len(gg):
        print(f"  {s:8s} worst 1d return {gg.ret.min():+.4f}  (kept)")

# proof the LUNA trap is defused
luna = panel[panel.Symbol.str.startswith('LUNA')]
if len(luna):
    print("\nLUNA eras after the fix:")
    for s, g in luna.groupby('Symbol'):
        r = g.sort_values('Date')['Close'].pct_change()
        print(f"  {s:10s} {len(g):5d} bars  {g.Date.min().date()} -> {g.Date.max().date()}  "
              f"max 1d return {r.max():+.2f}x")

## 6. Benchmark signals

Same definitions and the same **higher = go long** orientation as the equities pipeline, converted
from trading days to calendar days (crypto has no weekends — §1):

| Signal | Equities | Crypto | Source |
|---|---|---|---|
| `MOM` | t−252 → t−21 | **t−365 → t−30** | Jegadeesh & Titman (1993) |
| `STR` | −(21d return) | **−(30d return)** | Jegadeesh (1990) |
| `WSTR` | −(5d return) | **−(7d return)** | Lehmann (1990) |
| `MA{L}` | MA(L)/close | same | inputs to the Han–Zhou–Zhu trend factor |

TREND is **not** computed here — it needs the forward horizon, so it is fitted at scoring time in
`crypto_train.ipynb`, exactly as in the equities notebook.

Warm-up NaNs are retained on purpose.

In [ ]:
panel = panel.sort_values(['Symbol', 'Date']).reset_index(drop=True)
g = panel.groupby('Symbol')['Close']

panel['MOM']  = g.shift(MOM_SKIP) / g.shift(MOM_LB) - 1
panel['STR']  = -(panel['Close'] / g.shift(STR_LB) - 1)
panel['WSTR'] = -(panel['Close'] / g.shift(WSTR_LB) - 1)
for L in MA_LAGS:
    panel[f'MA{L}'] = panel.groupby('Symbol')['Close'].transform(
        lambda s, L=L: s.rolling(L).mean()) / panel['Close']

assert 'Symbol' in panel.columns, "Symbol column lost -- groupby regression"
print("signal NaNs retained (warm-up, by design):")
print({c: int(panel[c].isna().sum()) for c in ['MOM','STR','WSTR','MA3','MA200']})
print(f"\nrows: {len(panel):,}  (NOT dropped -- the CNN reads the image alone)")

## 7. Volatility regime — LOCKED specification, leak-free

Implemented exactly as pre-registered:

- per-coin realized vol = **std of daily log returns over a trailing 30-day window**;
- **shifted so the window ends at `t-1`** — the regime for the forward window starting at `t` uses
  only information known strictly before `t`;
- at each date the universe is ranked into **LOW / MID / HIGH** terciles;
- regimes **re-form each date**.

This is for **stratification only**. It is never a CNN input — the CNN sees the image and nothing
else. `min_periods=VOL_WIN` forces a complete window, so there is no partial-window leakage.

In [ ]:
panel['LogRet']   = panel.groupby('Symbol')['Close'].transform(lambda s: np.log(s).diff())
# rolling(30).std() at t covers t-29..t  ->  .shift(1) makes it cover t-30..t-1  (ends at t-1)
panel['RealVol30'] = panel.groupby('Symbol')['LogRet'].transform(
    lambda s: s.rolling(VOL_WIN, min_periods=VOL_WIN).std().shift(1))

print(f"RealVol30 defined on {panel.RealVol30.notna().sum():,} / {len(panel):,} rows")
print(f"  annualized vol (x sqrt({ANNUALIZER})) percentiles across the panel:")
av = panel['RealVol30'].dropna() * np.sqrt(ANNUALIZER)
print("   " + "  ".join(f"p{q}={av.quantile(q/100):.2f}" for q in [5, 25, 50, 75, 95]))

# ---- leak audit: RealVol30 at t must be reconstructible from returns strictly before t ----
_s = panel[panel.Symbol == 'BTC'].sort_values('Date').reset_index(drop=True)
i = 400
manual = _s['LogRet'].iloc[i-VOL_WIN:i].std()          # returns t-30..t-1, excludes t
stored = _s['RealVol30'].iloc[i]
print(f"\nleak audit (BTC row {i}): stored={stored:.8f} manual_t-30..t-1={manual:.8f} "
      f"-> {'PASS' if np.isclose(stored, manual) else 'FAIL'}")

## 8. Point-in-time universe + regime terciles

`DollarVol30` = trailing 30d **median** of `Close × Volume`, `.shift(1)` so it is known strictly
before `t`. At each date, assets are ranked by it and the **top 100** form the universe.

Median (not mean) for the same reason the equities project used it: one listing-day volume spike
should not buy a coin a permanent seat.

**This is a liquidity rank, not market cap** — see the header. Vol terciles are formed *within* the
universe on each date, so the regimes describe the tradeable cross-section.

In [ ]:
panel['DollarVol']   = panel['Close'] * panel['Volume']
panel['DollarVol30'] = panel.groupby('Symbol')['DollarVol'].transform(
    lambda s: s.rolling(LIQ_WIN, min_periods=10).median().shift(1))       # strictly past

panel['UnivRank']   = panel.groupby('Date')['DollarVol30'].rank(ascending=False, method='first')
panel['InUniverse'] = panel['UnivRank'].notna() & (panel['UnivRank'] <= TOP_N)

print(f"universe rows: {int(panel.InUniverse.sum()):,} / {len(panel):,} "
      f"({100*panel.InUniverse.mean():.1f}%)")

# ---- vol terciles, re-formed each date, within the universe ----
def _terc(s):
    if s.notna().sum() < 3:
        return pd.Series([None] * len(s), index=s.index, dtype=object)
    return pd.Series(pd.qcut(s.rank(method='first'), 3,
                             labels=['LOW','MID','HIGH']).astype(object), index=s.index)

u = panel[panel.InUniverse & panel.RealVol30.notna()]
# NOTE: initialise as OBJECT, not np.nan. `panel['VolRegime'] = np.nan` creates a float64
# column; writing 'LOW'/'MID'/'HIGH' into it raises under pandas 3
#   TypeError: Invalid value '[...]' for dtype 'float64'
# (pandas 2 silently upcast to object; pandas 3 refuses). Same trap for any string/categorical
# column built this way.
panel['VolRegime'] = pd.Series([None] * len(panel), index=panel.index, dtype='object')
panel.loc[u.index, 'VolRegime'] = u.groupby('Date')['RealVol30'].transform(_terc)

vc = panel.loc[panel.InUniverse, 'VolRegime'].value_counts(dropna=False)
print(f"\nvol regime counts within the universe:\n{vc.to_string()}")

chk = (panel[panel.InUniverse & panel.VolRegime.notna()]
       .groupby('VolRegime')['RealVol30'].median() * np.sqrt(ANNUALIZER))
print(f"\nmedian annualized vol by regime (must be ordered LOW < MID < HIGH):")
print(chk.reindex(['LOW','MID','HIGH']).round(3).to_string())

## 9. Honesty report — where the bias actually is

The equities analogue of this section had to confess that 429/703 de-indexed names were
unrecoverable and coverage drifted +34.9pp. **Crypto's numbers are better, but not clean**, and the
residual bias is real. State it; do not solve it with adjectives.

In [ ]:
print("=" * 74)
print("SURVIVORSHIP / COVERAGE HONESTY REPORT — CRYPTO")
print("=" * 74)

uni = panel[panel.InUniverse]
dead_syms = set(cand[cand.status == 'BREAK']['base'])
ever_uni  = set(uni['Symbol'].str.split('#').str[0])
dead_in_uni = ever_uni & dead_syms

print(f"assets ever in the top-{TOP_N} universe : {len(ever_uni):,}")
print(f"  ...that are now DELISTED (BREAK)     : {len(dead_in_uni):,}  <- RETAINED, with full history")
print(f"  ...still trading                     : {len(ever_uni - dead_syms):,}")
print(f"fetch failures                         : {len(failed):,}")
print(f"\nThis is the inverse of the equities panel: the dead names are PRESENT, not missing.")

print("\n" + "-" * 74)
print("UNIVERSE BREADTH BY YEAR (target 100; <20 cannot support deciles)")
print("-" * 74)
by_yr = uni.groupby(uni.Date.dt.year).agg(
    mean_names=('Symbol', lambda s: round(s.groupby(uni.loc[s.index, 'Date']).nunique().mean(), 1)),
    min_names=('Symbol', lambda s: s.groupby(uni.loc[s.index, 'Date']).nunique().min()),
    pct_BTCX=('Source', lambda s: round(100 * (s == 'BTCX').mean(), 1)))
print(by_yr.to_string())

print("\n" + "-" * 74)
print("EXTREME MOVES SURVIVING THE ERA-SPLIT (reported, NOT filtered)")
print("-" * 74)
# Returns MUST be computed on the full panel and restricted to genuinely CONSECUTIVE bars.
# Calling pct_change() on universe-filtered rows compares prices across membership gaps that
# can be months apart -- that bug reported a fake '+5762x' for COCOS, whose actual move that
# day was -9.4%. The universe is not a continuous series; the price panel is.
_p = panel.sort_values(['Symbol','Date'])
_step = _p.groupby('Symbol')['Date'].diff().dt.days
_p = _p.assign(ret1d=_p.groupby('Symbol')['Close'].pct_change().where(_step == 1))
ext = _p[_p.InUniverse & ((_p.ret1d > 5) | (_p.ret1d < -0.9))]
print(f"universe rows with a true consecutive-day return > +500% or < -90%: {len(ext):,}")
if len(ext):
    print(ext[['Date','Symbol','Close','ret1d','Source']]
          .sort_values('ret1d').to_string(index=False, float_format=lambda x: f"{x:,.5g}"))
_u = _p[_p.InUniverse & _p.ret1d.notna()]
print(f"\nuniverse 1d return range: {_u.ret1d.min():+.3f} .. {_u.ret1d.max():+.3f} "
      f"| p0.1 {_u.ret1d.quantile(.001):+.3f} | p99.9 {_u.ret1d.quantile(.999):+.3f}")
print("\n  What remains are REAL crypto moves (LUNA's collapse, OM's 2025 crash), not glitches:")
print("  units changes were removed structurally in §5 by cutting the series, not by clipping")
print("  returns. They are left in: silently winsorizing the tails of a long-short book is how")
print("  a null result gets laundered into a Sharpe.")

print("\n" + "=" * 74)
print("RESIDUAL BIAS — what this test still CANNOT support")
print("=" * 74)
print('''
1. BINANCE-LISTING SURVIVORSHIP (the real one, and it is unfixable from free data).
   The universe is 'top 100 of what Binance listed', NOT 'top 100 of all crypto'. Listing is
   conditional on a coin surviving long enough to BE listed, so coins that pumped and died on a
   DEX -- or died on Bittrex/Poloniex in 2018 -- never enter the panel. The absent names skew to
   catastrophic losers, the SAME direction as the equities bias: a missing loser flatters a short
   leg. This inflates the long-short result by an unknown amount.

2. DELISTING TRUNCATION. When Binance delists a coin its history simply ends. The coin may keep
   falling elsewhere. Forward returns spanning a delisting are undefined and get dropped, so the
   final leg down is invisible -- again flattering the short leg.

3. LIQUIDITY RANK != MARKET CAP. The universe is ranked by dollar volume because no free
   point-in-time market cap exists. It over-weights high-turnover coins.

4. CROSS-RATE APPROXIMATION. Pre-2020 rows are BTC-quoted x BTCUSDT: the close is exact, the
   intraday high/low are approximate (see the pct_BTCX column above for exactly how much of each
   year this touches).

IF THE CNN SHOWS NO EDGE: 'no signal' and 'survivorship artifact' are EQUALLY LIKELY and neither
can be ruled out from this data. Say exactly that -- do not pick the flattering one.
''')

## 10. Save

Schema for `crypto_train.ipynb`. Deliberate deviations from the equities parquet:

- `Symbol` replaces `Ticker` (these are assets, not tickers, and eras carry a `#n` suffix);
- `InUniverse` replaces `InIndex`;
- **no `market_cap`** — it does not exist free; `DollarVol30` + `UnivRank` carry the ranking, named
  so they cannot be mistaken for it;
- `RealVol30` / `VolRegime` are new (Stage 4 stratification, analysis only — never a CNN input);
- `Source` records USDT vs BTC-cross per row, so the approximation stays auditable.

In [ ]:
cols = (['Date','Symbol','Open','High','Low','Close','Volume','MOM','STR','WSTR'] + MA_COLS
        + ['InUniverse','UnivRank','DollarVol30','RealVol30','VolRegime','Source'])
out = panel[cols].sort_values(['Symbol','Date']).reset_index(drop=True)
out['Date'] = pd.to_datetime(out['Date'])
out['InUniverse'] = out['InUniverse'].astype(bool)
out.to_parquet(OUT_FILE, index=False)

print(f"saved {OUT_FILE}")
print(f"  {len(out):,} rows | {out.Symbol.nunique():,} assets | "
      f"{out.Date.min().date()} -> {out.Date.max().date()} | "
      f"{os.path.getsize(OUT_FILE)/1e6:.1f} MB")
print(f"  universe rows: {int(out.InUniverse.sum()):,}")
print(f"\ndtypes:\n{out.dtypes.to_string()}")
display(out[out.InUniverse].head())